Async, Concurrency & Event Loop Internals

How FastAPI handles multiple users and multiple I/O operations withoit unnecessarily blocking ther server

1. Synchronous vs asynchronous programming.
2. Concurrency vs parallelism.
3. Coroutines, event loops, tasks, and await.
4. How FastAPI executes def versus async def.
5. Sequential vs concurrent API calls.
6. asyncio.gather() and TaskGroup.
7. Timeouts, cancellation, and error propagation.
8. Semaphores, connection pools, and backpressure.
9. Blocking LLM SDKs and CPU-heavy inference.
10. A complete asynchronous FastAPI AI workflow.

Synchronous

In [1]:
import time
def get_vendor():
    print("fetching the vendor details.....")
    time.sleep(2)
    return "Aira AI"

def get_invoice():
    print("fetching the invoice details.....")
    time.sleep(3)
    return "INV-101"

def main():
    vendor=get_vendor()
    invoice=get_invoice()
    print(vendor,invoice)

main()

fetching the vendor details.....
fetching the invoice details.....
Aira AI INV-101


Asynchronous


In [2]:
import asyncio

async def get_vendor():
    print("Fetching vendor...")
    await asyncio.sleep(2)
    return "Aira AI"

async def get_invoice():
    print("Fetching invoice...")
    await asyncio.sleep(3)
    return "INV-101"

async def main():
    vendor, invoice = await asyncio.gather(
        get_vendor(),
        get_invoice()
    )

    print(vendor, invoice)


await main()

Fetching vendor...
Fetching invoice...
Aira AI INV-101


Coroutines, Event Loops and Tasks

Coroutine - A coroutine is a function execution that can suspend and resume.

Whenever we write an async function
That will be coroutine function
and when calling it'll be coroutine object

In [3]:
async def generate():
    return "Hello"

# generate()
print(await generate())
print(type(generate()))


Hello
<class 'coroutine'>


C:\Users\venkat\AppData\Local\Temp\ipykernel_32812\1097306589.py:6: RuntimeWarning: coroutine 'generate' was never awaited
  print(type(generate()))


Event Loop - Is a scheduler that coordinates asynchronous tasks.

In [4]:
async def task_a():
    print("A started")
    await asyncio.sleep(2)
    print("A finished")


async def task_b():
    print("B started")
    await asyncio.sleep(1)
    print("B finished")

async def main():
    await asyncio.gather(task_a(), task_b())

await main()

A started
B started
B finished
A finished


When a coroutine reaches an await on an incomplete asynchronous operation, it can suspend. The event loop then runs other ready tasks until the suspended operation can continue.
This is called cooperative multitasking.
It differs from operating-system preemptive threading because coroutines generally cooperate by yielding at suspension points.

Task - A coroutine object describes asynchronous work. An asyncio.Task schedules that coroutine for execution on the event loop.

In [5]:
async def main():
    task = asyncio.create_task(get_vendor())

    print("Other work")

    vendor = await task
    print(vendor)
get_vendor()
await main()

C:\Users\venkat\AppData\Local\Temp\ipykernel_32812\2212023312.py:8: RuntimeWarning: coroutine 'get_vendor' was never awaited
  get_vendor()


Other work
Fetching vendor...
Aira AI


Timeouts, Cancellation and Semaphores

In [6]:
async def call_model():
    await asyncio.sleep(10)
    return "Completed"


async def main():
    try:
        async with asyncio.timeout(3):
            result = await call_model()

        print(result)

    except TimeoutError:
        print("LLM request timed out")
    # result=await call_model()
    # print(result)
await main()

LLM request timed out


Strcutured concurrency with TaskGroup

If one task fails with an ordinary exception, the remaining tasks are normally cancelled and the errors are propagated, potentially inside an ExceptionGroup.

In [1]:
import asyncio


async def run_workflow():
    async with asyncio.TaskGroup() as tg:
        vendor_task = tg.create_task(get_vendor())
        policy_task = tg.create_task(get_policy())

    return {
        "vendor": vendor_task.result(),
        "policy": policy_task.result()
    }

Cancellation cleanup

In [31]:
async def process_document():
    print("Started")

    try:
        await asyncio.sleep(10)
    finally:
        print("Cleaning up")

Seamaphore

A semaphore limits the number of tasks allowed to enter a particular section of code simultaneously.

In [33]:
# import asyncio
# llm_semaphore = asyncio.Semaphore(5)

# async def call_llm(prompt: str):
#     async with llm_semaphore:
#         return await real_async_llm_call(prompt)

Full Workflow of FastAPI

A user submits an invoice ID. Our service concurrently fetches vendor information, payment history, and payment policy, then assembles context for an AI agent.

In [38]:

import asyncio
import time
from contextlib import asynccontextmanager
from typing import Annotated

from fastapi import (
    FastAPI,
    HTTPException,
    Path,
    Request,
)
from pydantic import BaseModel, Field


# Limit concurrent invoice workflow executions
workflow_semaphore = asyncio.Semaphore(5)


class InvoiceRequest(BaseModel):
    question: str = Field(
        min_length=1,
        max_length=2000
    )


class InvoiceResponse(BaseModel):
    invoice_id: int
    question: str
    context: dict
    answer: str
    elapsed_seconds: float


async def fetch_vendor(invoice_id: int):
    await asyncio.sleep(2)

    return {
        "vendor_id": 22,
        "name": "Aira AI"
    }


async def fetch_payment_history(invoice_id: int):
    await asyncio.sleep(3)

    return {
        "invoice_id": invoice_id,
        "past_due_count": 2
    }


async def fetch_payment_policy(invoice_id: int):
    await asyncio.sleep(1)

    return {
        "payment_terms": "Net 30",
        "approval_required": True
    }


async def generate_answer(question: str, context: dict):
    # Mock LLM call
    await asyncio.sleep(1)

    return (
        f"Question: {question}. "
        f"Vendor: {context['vendor']['name']}. "
        f"Payment terms: "
        f"{context['policy']['payment_terms']}."
    )


app = FastAPI(
    title="Finance AI Async API"
)


@app.get("/health")
async def health():
    return {"status": "ok"}


@app.post(
    "/invoices/{invoice_id}/analyze",
    response_model=InvoiceResponse
)
async def analyze_invoice(
    invoice_id: Annotated[int, Path(gt=0)],
    payload: InvoiceRequest
):
    start = time.perf_counter()

    try:
        async with asyncio.timeout(8):
            async with workflow_semaphore:

                # Independent upstream calls
                vendor, history, policy = await asyncio.gather(
                    fetch_vendor(invoice_id),
                    fetch_payment_history(invoice_id),
                    fetch_payment_policy(invoice_id)
                )

                context = {
                    "vendor": vendor,
                    "history": history,
                    "policy": policy
                }

                # Depends on retrieved context
                answer = await generate_answer(
                    payload.question,
                    context
                )

    except TimeoutError:
        raise HTTPException(
            status_code=504,
            detail="Invoice analysis timed out"
        )

    elapsed = time.perf_counter() - start

    return InvoiceResponse(
        invoice_id=invoice_id,
        question=payload.question,
        context=context,
        answer=answer,
        elapsed_seconds=round(elapsed, 3)
    )


# Interview Answer: Designing a FastAPI AI Agent for 100 Concurrent Users

AUDITORIA.AI · AI BACKEND SYSTEM DESIGN

This is a strong AI Engineer interview question because it tests whether you understand FastAPI concurrency, async I/O, database connection pools, LLM rate limits, timeouts, failure handling, and production scalability.

## 1. The answer I'd give in the interview (90 seconds)

> I'd build the FastAPI endpoint using `async def`, with asynchronous clients for Qdrant, PostgreSQL, and the LLM provider.
>
> When 100 requests arrive, I wouldn't execute all operations without limits. I'd first authenticate the users, enforce per-user or per-tenant rate limits, and then apply concurrency limits to protect downstream services.
>
> Within each request, I'd query Qdrant and PostgreSQL concurrently using `asyncio.gather()` or `TaskGroup`, assuming neither operation depends on the other. Once both complete, I'd construct the prompt and call the LLM.
>
> I'd reuse async clients and database connection pools initialized during application startup, instead of creating new connections for each request.
>
> I'd apply separate timeouts to retrieval, database operations, and LLM generation, plus an overall request deadline. I'd handle provider throttling, transient failures, and cancellations, using bounded retries with exponential backoff and jitter only when safe.
>
> If the system reaches its LLM concurrency or token quota, I'd use backpressure: queue requests with bounded waiting or return HTTP 429 or 503 with an appropriate retry policy.
>
> Finally, I'd add structured logging, request IDs, distributed tracing, and metrics for p95 latency, database pool utilization, LLM throughput, timeout rates, and upstream failures. I'd scale the API and inference capacity independently based on those measurements.

That's a solid answer. Now let's understand the architecture and implementation so you can defend every decision.

# 2. High-level architecture

The LLM call starts after retrieval context is available. Qdrant and PostgreSQL execute concurrently only if their operations are independent.

## 3. How concurrency actually works

Say each request has the following approximate service times:

| Operation        | Time     |
| ---------------- | -------- |
| Qdrant retrieval | 150 ms   |
| PostgreSQL query | 100 ms   |
| LLM generation   | 3,000 ms |

If we execute every step sequentially:

\\[ T = 150 + 100 + 3000 = 3250\text{ ms} \\]

If Qdrant and PostgreSQL execute concurrently:

\\[ T \approx \max(150,100)+3000=3150\text{ ms} \\]

Notice something important: parallelizing retrieval only saves approximately 100 ms in this example.

The LLM remains the dominant latency.

Illustrative request latency by component

0 ms800 ms1.6K ms2.4K ms3.2K msQdrantPostgreSQLLLM

Hypothetical single-request service times, not benchmarks.

So the real question isn't just how to handle 100 HTTP requests.

It's how to control the number of simultaneous LLM operations while still keeping the FastAPI event loop responsive.

For example, suppose the provider can sustain 10 concurrent generations and each takes three seconds. With 100 LLM requests already accepted and no other constraints, completion could occur in approximately 10 waves, taking around 30 seconds for the last wave.

That's a simplified capacity model, not a guarantee. Token quotas, variable output lengths, provider queuing, and request overhead can change the result.

# 4. Implementation: FastAPI concurrency control

Let's implement the critical part using Python. We'll simulate Qdrant, PostgreSQL, and an LLM so you can run the example without external services.

```
@asynccontextmanagerasync def lifespan(app: FastAPI):    # Example per-process limits, not universal defaults.    app.state.llm_limit = asyncio.Semaphore(10)    app.state.retrieval_limit = asyncio.Semaphore(30)    # Real system: initialize async database, Qdrant,    # and HTTP clients here; close them on shutdown.    yieldapp = FastAPI(lifespan=lifespan)async def retrieve_qdrant(question: str):    await asyncio.sleep(0.15)    return ["Invoice policy", "Payment policy"]async def query_postgres():    await asyncio.sleep(0.10)    return {"customer_id": 101, "account": "active"}async def call_llm(question: str, documents: list[str]):    await asyncio.sleep(3)    return f"Generated answer for: {question}"@app.post("/agent/query", response_model=AgentResponse)async def query_agent(payload: AgentRequest):    try:        # Total workflow deadline        async with asyncio.timeout(12):            # Limit retrieval work            async with app.state.retrieval_limit:                async with asyncio.TaskGroup() as tg:                    docs_task = tg.create_task(                        retrieve_qdrant(payload.question)                    )                    db_task = tg.create_task(                        query_postgres()                    )                documents = docs_task.result()                customer = db_task.result()            # Limit expensive LLM calls            async with app.state.llm_limit:                answer = await call_llm(                    payload.question,                    documents                )            return AgentResponse(                answer=answer,                documents=documents,                customer=customer            )    except TimeoutError:        raise HTTPException(            status_code=504,            detail="Agent workflow timed out"        )
```

### What happens when 100 requests arrive?

Initially, up to 30 requests can enter our retrieval section, and up to 10 can enter the LLM section. Others wait for a semaphore permit.

These numbers are merely illustrative and must be chosen based on measured backend capacity.

There are also three important limitations in this demonstration:

- It uses simulated operations rather than real clients.
- Waiting for semaphores is bounded only by the overall 12-second timeout; under 100 simultaneous requests, many may time out.
- Unexpected exceptions from `TaskGroup` can propagate as an exception group. Production code needs explicit upstream-error mapping and cancellation-aware cleanup.

For a production implementation, I would add a short maximum admission wait or a bounded queue, returning `503 Service Unavailable` when the application cannot safely accept more work.

## 5. Timeouts and error-handling strategy

I'd use a hierarchy of timeouts rather than one timeout everywhere.

| Component      | Example budget | Failure behavior                                 |
| -------------- | -------------- | ------------------------------------------------ |
| Qdrant         | 2 seconds      | Retry once if transient; fail or degrade if safe |
| PostgreSQL     | 2 seconds      | Return controlled error; release session         |
| LLM            | 8 seconds      | Timeout or fallback where appropriate            |
| Total request  | 12 seconds     | Cancel remaining work; return 504                |
| Admission wait | 500 ms         | Reject overload with 503                         |

These are example engineering budgets, not guaranteed optimal settings.

For an LLM streaming endpoint, the time-to-first-token and inter-token read timeouts may need different treatment from a fixed total generation deadline.

### Error-handling matrix

| Failure                          | Handling                                 |
| -------------------------------- | ---------------------------------------- |
| Invalid input                    | 422                                      |
| Invalid/missing authentication   | 401                                      |
| Unauthorized tenant access       | 403 or non-disclosing 404                |
| User request-rate limit exceeded | 429, optionally `Retry-After`            |
| Qdrant unavailable               | 503 or controlled degraded retrieval     |
| PostgreSQL unavailable           | 503                                      |
| LLM provider returns 429         | Bounded wait/retry or propagate overload |
| Upstream LLM error               | 502 or 503                               |
| Upstream timeout                 | 504                                      |
| Client disconnects               | Cancel work where possible and clean up  |

I wouldn't retry every failure. Authentication failures, schema errors, or known permanent errors shouldn't be retried. For finance actions, retries must also respect idempotency.

## 6. PostgreSQL session safety

One subtle concurrency bug can appear when you share one SQLAlchemy `AsyncSession` across multiple concurrent tasks.

SQLAlchemy explicitly warns that an individual `AsyncSession` is not safe for use by multiple concurrent tasks.&#x20;

[image](https://www.google.com/s2/favicons?domain=https://docs.sqlalchemy.org\&sz=32)

SQLAlchemy 2.1 Documentation



For example, avoid:

```
# Unsafe if both functions concurrently use the same sessionawait asyncio.gather(    get_customer(db_session),    get_invoice(db_session))
```

Instead, separate concurrent database operations should use independent sessions, or queries should be executed sequentially within a single transaction when they must share transactional state.

A connection pool can still be shared safely through the configured database engine. The pool controls access to database connections; the ORM session represents mutable transactional state.

For Qdrant, I'd use its asynchronous client, and for external HTTP calls I'd reuse a pooled `httpx.AsyncClient` through FastAPI lifespan.&#x20;

[image](https://www.google.com/s2/favicons?domain=https://qdrant.tech\&sz=32)

Qdrant

+1



## 7. Production concerns beyond async

At 100 concurrent users, I'd also consider:

- Authentication and tenant isolation: Verify each user's access to PostgreSQL records and Qdrant collections or payload filters.
- Prompt security: Treat retrieved documents as untrusted data and restrict the AI agent's tool permissions.
- Token budgets: Enforce maximum input/output tokens and provider-specific requests-per-minute and tokens-per-minute quotas.
- Backpressure: Avoid unbounded request queues that consume memory and cause timeouts.
- Circuit breakers: Stop repeatedly calling a consistently failing provider for a bounded recovery period.
- Observability: Trace Qdrant retrieval, SQL execution, prompt construction, model queue time, time-to-first-token, and generation.
- Horizontal scaling: Use shared rate-limit state when multiple API workers or replicas are deployed.

For particularly long-running agent workflows, I'd return HTTP 202 with a job ID and process the work using a durable worker queue. This avoids holding many client connections open while execution takes minutes.

## 8. How I would conclude in the interview

> "My objective wouldn't simply be to allow 100 concurrent requests. It would be to maintain predictable latency, bounded resource usage, tenant isolation, and reliable failure handling under that load.
>
> FastAPI's async capabilities help us overlap network I/O, but connection pools, semaphores, deadlines, rate limits, and upstream capacity ultimately determine how much traffic the system can safely handle.
>
> I'd benchmark realistic workloads, monitor p95 and p99 latency, and adjust concurrency limits based on measured Qdrant, PostgreSQL, and LLM throughput."

One sentence to remember: Async helps with efficient waiting; concurrency limits protect shared resources; queues handle excess demand; and observability tells us where the system actually slows down.

For AuditoriaAI, especially emphasize idempotency, auditability, tenant isolation, and avoiding duplicate financial actions. Those are the differences between building an impressive LLM demo and engineering a dependable enterprise AI system.

Our AI backend needs to:
1. Authenticate the user and check their rate limit.
2. Retrieve relevant documents from Qdrant.
3. Fetch invoice information from PostgreSQL.
4. Combine the information into a prompt.
5. Send the prompt to an LLM.
6. Return the generated answer.